# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/victorydavid-lab/Victory/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [21]:
print("Colab is working!")

Colab is working!


In [22]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Token found:", bool(HF_TOKEN))

Token found: True


In [23]:
from huggingface_hub import whoami

print(whoami(token=HF_TOKEN))

{'type': 'user', 'id': '6a76589d64efa13649a0104e', 'name': 'VictoryDavid', 'fullname': 'Victory Chima-David', 'email': 'victorytitilayod@gmail.com', 'emailVerified': True, 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1788220800, 'isPro': False, 'avatarUrl': '/avatars/090e164d568f2ab7e8e2f131c65b4836.svg', 'orgs': [], 'auth': {'type': 'access_token', 'accessToken': {'displayName': 'ScholarAI', 'role': 'read', 'createdAt': '2026-08-07T22:17:56.181Z'}}}


In [24]:
# FlyRank Week 3 — data connection

%pip -q install duckdb huggingface_hub

import os
import getpass
import duckdb

# Get Hugging Face token from Colab Secret
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    HF_TOKEN = None

if not HF_TOKEN:
    HF_TOKEN = getpass.getpass("Paste your Hugging Face READ token: ")

os.environ["HF_TOKEN"] = HF_TOKEN

# Connect DuckDB
con = duckdb.connect()

# FlyRank dataset
REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "clients": f"read_parquet('{REL}/dim_clients.parquet')",
    "content": f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily": f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
    "fact_daily_sample": f"read_parquet('{REL}/fact_content_daily_performance/month=2026-06/*.parquet')",
    "query_90d": f"read_parquet('{REL}/fact_content_query_90d.parquet')",
}

print("FlyRank connection ready.")
print(TABLES)

FlyRank connection ready.
{'clients': "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_clients.parquet')", 'content': "read_parquet('hf://datasets/FlyRank/internship-warehouse/dim_content.parquet')", 'fact_daily': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')", 'fact_daily_sample': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-06/*.parquet')", 'query_90d': "read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_query_90d.parquet')"}


In [25]:
# Give DuckDB the Hugging Face token

con.execute("""
CREATE OR REPLACE SECRET hf_token (
    TYPE HUGGINGFACE,
    TOKEN ?
)
""", [HF_TOKEN])

print("Hugging Face authentication configured.")

Hugging Face authentication configured.


In [26]:
test = con.sql(f"""
SELECT *
FROM {TABLES['fact_daily']}
LIMIT 5
""").df()

test

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-03-01,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,True,False,True,<NA>,20,0,67,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
1,2026-03-01,client_73cda7b4e4f265ea,content_05597932fe4da067,True,False,True,<NA>,1,0,0,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
2,2026-03-01,client_73cda7b4e4f265ea,content_7a105f548d9c6916,True,False,True,<NA>,125,1,616,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
3,2026-03-01,client_73cda7b4e4f265ea,content_905aa32a0230694e,True,False,True,<NA>,7,0,28,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03
4,2026-03-01,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,True,False,True,<NA>,11,0,25,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-03


## 1. Unit of analysis + time window

1) Unit of analysis: One row represents one content item for one client on one reporting date. This is supported by the March 2026 data: the 9,841,378 rows contain 9,841,378 unique client-content-date combinations.

2) Table(s): I will use the fact_content_daily_performance table as my main source for content-level search performance. It provides daily performance data for each client and content item. I will use the March 2026 panel for the initial verification and feature work.

3) Time window: The initial analysis uses March 1–31, 2026. The March panel contains 9,841,378 rows. For later modeling, the feature and outcome windows will be separated so that information from the future is not used to create features.

4) What I will predict/rank: I will work toward ranking content items by their search-performance opportunity — identifying content that may have an opportunity to improve its organic search performance. The eventual label/proxy will be defined using a future outcome window rather than the same data used to create the features.

5) Deliberately excluded: I will exclude rows where GSC data is unavailable from GSC-based performance features rather than treating missing values as zero. In March, only 3,611,061 of 9,841,378 rows have GSC data available, so assuming missing GSC data means zero performance could produce misleading features.

In [27]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

Features: gsc_impressions, gsc_clicks, gsc_avg_position — these describe how each content item is performing in Google Search and can be calculated from the information available at the decision moment.

Label: A future-period search-performance outcome, such as whether a content item improves its average search position or increases impressions. The label will be created later from a separate future outcome window so that it does not leak into the features.

Context: client_hash_id, content_hash_id, and report_date — these identify the client, content item, and date, allowing performance to be grouped and compared correctly.

Excluded: GA4 fields and AI-referral fields for this first analysis because the March slice shows that GA4 data is not consistently available. I will also exclude client_hash_id and content_hash_id from predictive features because they are identifiers rather than measures of search performance.

In [28]:
# Show the exact columns in the March fact table

columns = con.sql(f"""
SELECT *
FROM {TABLES['fact_daily']}
LIMIT 0
""").df().columns.tolist()

for i, column in enumerate(columns, 1):
    print(f"{i}. {column}")

1. report_date
2. client_hash_id
3. content_hash_id
4. client_has_gsc
5. client_has_ga4
6. gsc_data_available
7. ga4_data_available
8. gsc_impressions
9. gsc_clicks
10. gsc_sum_position
11. gsc_avg_position
12. ga4_pageviews
13. ga4_sessions
14. ga4_users
15. ga4_engaged_sessions
16. ga4_total_engagement_sec
17. sessions_organic
18. sessions_direct
19. sessions_referral
20. sessions_social
21. sessions_paid
22. sessions_ai
23. ai_chatgpt
24. ai_perplexity
25. ai_gemini
26. ai_copilot
27. ai_claude
28. ai_meta
29. ai_other
30. scroll_events
31. month


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [29]:
# Query 1 — Verify the grain
# We expect one row per client + content item + reporting date.

grain_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT client_hash_id || '|' || content_hash_id || '|' || CAST(report_date AS VARCHAR)) AS unique_client_content_date
FROM {TABLES['fact_daily']}
""").df()

grain_check


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,unique_client_content_date
0,9841378,9841378


Result: The March 2026 panel contains 9,841,378 rows and 9,841,378 unique client-content-date combinations. Because the counts match, there are no duplicate rows at this grain. This supports the definition that one row represents one content item for one client on one reporting date.

In [30]:
# Query 2 — Row count and date span

date_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    MIN(report_date) AS start_date,
    MAX(report_date) AS end_date
FROM {TABLES['fact_daily']}
""").df()

date_check

,row_count,start_date,end_date
0,9841378,2026-03-01,2026-03-31


Result: The March 2026 panel contains 9,841,378 rows and covers the full period from March 1 to March 31, 2026. This provides a complete one-month panel for the verification and initial feature work.

In [31]:
# Query 3 — Verify GSC availability

availability_check = con.sql(f"""
SELECT
    COUNT(*) AS total_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS available_rows,
    COUNT(*) FILTER (WHERE gsc_data_available IS NOT TRUE) AS unavailable_rows
FROM {TABLES['fact_daily']}
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,available_rows,unavailable_rows
0,9841378,3611061,6230317


Result: Of the 9,841,378 rows in the March 2026 panel, 3,611,061 rows have GSC data available, while 6,230,317 rows do not. The availability check uses IS TRUE as required. Therefore, GSC performance fields are only available for a subset of the panel and should not be assumed to be present for every row.

## 4. Data limits

Data limits: This dataset can show observed search and traffic performance, but it cannot by itself prove that a particular feature caused a change in rankings or traffic. History is also unbalanced across clients and content items, so some items have more complete historical data than others. Some early rows are GSC-only because GA4 data is unavailable, which limits comparisons involving GA4. Finally, overlapping time windows can cause information from nearby periods to appear in both features and outcomes if the windows are not separated carefully. For this reason, I will use separate past and future windows when creating labels and avoid treating correlation as causation.

In [32]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.